# Vigil — Model Eğitimi v2 (Google Colab)
10M satır TII-SSRC-23 + ev trafiği fine-tuning. Colab'da çalıştırılır.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import glob
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google.colab import files

In [ ]:
# Kaggle API ile veri indir
!pip install kaggle -q

from google.colab import files
import json

print('kaggle.json yukle (Kaggle > Settings > API > Create New API Token)')
files.upload()

with open('kaggle.json') as f:
    cfg = json.load(f)
assert 'username' in cfg and 'key' in cfg, 'kaggle.json hatali format!'
print(f'Kaggle kullanici: {cfg["username"]}')

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d daniaherzalla/tii-ssrc-23 -p /content/data/ --unzip -q
csv_files = [f for f in glob.glob('/content/data/**/*.csv', recursive=True)
             if 'sample_data' not in f]
assert csv_files, 'CSV bulunamadi!'
TRAIN_PATH = csv_files[0]
print(f'Kullanilacak dosya: {TRAIN_PATH}')

In [ ]:
# (Opsiyonel) Ev trafiği fine-tuning verisi yükle
# src/capture_home.py ile yakaladığınız home_traffic.csv dosyasını yükleyin
# Yoksa bu hücreyi atlayın
print('home_traffic.csv yukleyin (yoksa Cancel a basin veya hucreyi atlayin)')
HOME_TRAFFIC_PATH = None
try:
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        HOME_TRAFFIC_PATH = f'/content/{fname}'
        print(f'Ev trafigi yuklendu: {HOME_TRAFFIC_PATH}')
    else:
        print('Ev trafigi yüklenmedi, sadece TII-SSRC-23 kullanilacak.')
except Exception:
    print('Ev trafigi yuklenmedi.')

In [ ]:
LABEL_COL = 'Label'
DROP_COLS = ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Traffic Type', 'Traffic Subtype']
TEST_SIZE = 0.2
RANDOM_STATE = 42
# Ucretsiz Colab icin 4M onerilir (~9GB RAM). Colab Pro icin 10M yapabilirsiniz.
SAMPLE_N = 4_000_000

In [ ]:
def load_optimized(path, sample_n):
    """Chunked loading ile RAM tasarrufu yaparak veri yukle."""
    sample = pd.read_csv(path, nrows=1000, on_bad_lines='skip')
    dtypes = {}
    for col in sample.select_dtypes(include='float64').columns:
        dtypes[col] = 'float32'
    for col in sample.select_dtypes(include='int64').columns:
        dtypes[col] = 'int32'

    chunks = []
    rows_loaded = 0
    chunk_size = 500_000

    for chunk in pd.read_csv(path, low_memory=False, on_bad_lines='skip',
                              dtype=dtypes, chunksize=chunk_size):
        chunks.append(chunk)
        rows_loaded += len(chunk)
        print(f'  Yuklendi: {rows_loaded:,} satir', end='\r')
        if rows_loaded >= sample_n:
            break

    df = pd.concat(chunks, ignore_index=True)
    print(f'\nToplam yuklenen: {len(df):,} satir')
    return df

print('Ana veri seti yukleniyor...')
df = load_optimized(TRAIN_PATH, SAMPLE_N)
print(f'Ham shape: {df.shape}')
print(f'RAM: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
df = df.replace([np.inf, -np.inf], np.nan).dropna()
print(f'Temizlendikten sonra: {df.shape}')
print(df[LABEL_COL].value_counts())

In [ ]:
# Ev trafiği varsa ekle
if HOME_TRAFFIC_PATH:
    print('Ev trafigi ekleniyor...')
    home_df = pd.read_csv(HOME_TRAFFIC_PATH, low_memory=False, on_bad_lines='skip')
    home_df = home_df.replace([np.inf, -np.inf], np.nan).dropna()

    # Sadece modelde olan sutunlari tut
    common_cols = [c for c in home_df.columns if c in df.columns]
    home_df = home_df[common_cols]

    # Eksik sutunlari sifirla
    for col in df.columns:
        if col not in home_df.columns:
            home_df[col] = 0.0

    home_df = home_df[df.columns]  # Sutun siralamasi esitle

    # Ana veriye ekle — ev trafiğini 5x tekrarla (dengeli temsil icin)
    home_repeated = pd.concat([home_df] * 5, ignore_index=True)
    df = pd.concat([df, home_repeated], ignore_index=True)
    print(f'Ev trafigi eklendikten sonra: {df.shape}')
    print(df[LABEL_COL].value_counts())
else:
    print('Ev trafigi yok, sadece TII-SSRC-23 kullaniliyor.')

In [ ]:
label_encoder = LabelEncoder()
feature_cols = [c for c in df.columns if c != LABEL_COL]

X = df[feature_cols].values
y = label_encoder.fit_transform(df[LABEL_COL])

del df  # RAM bosalt
import gc; gc.collect()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'X_train: {X_train.shape}')
print(f'Classes: {label_encoder.classes_}')

In [ ]:
model = RandomForestClassifier(
    n_estimators=150,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    max_depth=30,
    min_samples_leaf=5,
    verbose=1,
)
print('Egitim basliyor...')
model.fit(X_train, y_train)
print('Egitim tamamlandi.')

In [ ]:
y_pred = model.predict(X_test)
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Gercek')
plt.xlabel('Tahmin')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(12, 5))
importances.head(20).plot(kind='bar', color='teal')
plt.title('Feature Importance (Top 20)')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

In [ ]:
artifacts = {
    'model': model,
    'scaler': scaler,
    'label_encoder': label_encoder,
    'feature_cols': feature_cols,
}
joblib.dump(artifacts, 'model.pkl')
print('model.pkl kaydedildi.')
files.download('model.pkl')
files.download('confusion_matrix.png')
files.download('feature_importance.png')
print('Indirme tamamlandi. model.pkl -> models/ klasorune koy.')